# 🔬 Étape 5 — Tests de robustesse

> **Méthodologie d'audit LLM v1.0** — par Hanen Mizouni

## 🎯 Objectif

Mesurer la stabilité du LLM face aux perturbations textuelles, dans le temps, et en cas d'usages aux limites.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

AUDIT_NAME = "audit_demo_chatbot_rh"
OUTPUT_DIR = Path(f"./output/{AUDIT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

## 5.1 Tests de robustesse aux perturbations textuelles

In [ ]:
def generate_perturbations(prompt: str) -> dict:
    """
    Génère des variations d'un prompt pour tester la robustesse.
    """
    return {
        "original": prompt,
        "uppercase": prompt.upper(),
        "lowercase": prompt.lower(),
        "with_typos": prompt.replace("e", "é").replace("a", "à")[:len(prompt)],
        "with_spaces": "  ".join(prompt.split()),
        "no_punctuation": prompt.replace(".", "").replace(",", "").replace("?", ""),
        "with_emojis": prompt + " 😊",
    }

test_prompts = [
    "Bonjour, je cherche un emploi d'ingénieur.",
    "Quelle est la culture d'entreprise chez vous ?",
    "Pouvez-vous me donner des conseils pour mon CV ?"
]

# Simulation des résultats (en prod : appels API)
robustness_results = []
for prompt in test_prompts:
    perturbations = generate_perturbations(prompt)
    for variation_type, variation_text in perturbations.items():
        # Simulation : taux de cohérence selon le type de perturbation
        coherence = {
            "original": 1.0,
            "uppercase": 0.85,
            "lowercase": 0.95,
            "with_typos": 0.78,
            "with_spaces": 0.92,
            "no_punctuation": 0.88,
            "with_emojis": 0.94
        }.get(variation_type, 0.9)
        
        robustness_results.append({
            "prompt_id": prompt[:30] + "...",
            "variation": variation_type,
            "coherence_score": coherence + np.random.normal(0, 0.05)
        })

df_robustness = pd.DataFrame(robustness_results)

# Visualisation
fig, ax = plt.subplots(figsize=(12, 6))
df_pivot = df_robustness.pivot_table(
    index="variation", columns="prompt_id", values="coherence_score", aggfunc="mean"
)
df_pivot.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Cohérence des réponses face aux perturbations", fontweight="bold")
ax.set_ylabel("Score de cohérence (0-1)")
ax.set_ylim(0, 1.1)
ax.axhline(0.9, color="red", linestyle="--", label="Seuil acceptable (0.9)")
ax.legend(title="Prompt", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

# Identification des perturbations problématiques
avg_by_perturbation = df_robustness.groupby("variation")["coherence_score"].mean().sort_values()
print("\n🚩 Perturbations les plus problématiques :\n")
for var, score in avg_by_perturbation.items():
    statut = "🔴" if score < 0.8 else "🟠" if score < 0.9 else "🟢"
    print(f"  {statut} {var:20s} : {score:.2f}")

## 5.2 Tests de cohérence sémantique (50 itérations)

In [ ]:
# Simulation : 50 réponses au même prompt
n_iterations = 50

# Variance simulée des longueurs de réponse
lengths = np.random.normal(loc=250, scale=35, size=n_iterations)
sentiments = np.random.normal(loc=0.7, scale=0.1, size=n_iterations)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(lengths, bins=15, color="#3498db", edgecolor="black")
axes[0].axvline(lengths.mean(), color="red", linestyle="--", label=f"Moyenne: {lengths.mean():.0f}")
axes[0].set_title(f"Distribution longueur réponses ({n_iterations} itérations)", fontweight="bold")
axes[0].set_xlabel("Mots")
axes[0].legend()

axes[1].hist(sentiments, bins=15, color="#27ae60", edgecolor="black")
axes[1].axvline(sentiments.mean(), color="red", linestyle="--", label=f"Moyenne: {sentiments.mean():.2f}")
axes[1].set_title(f"Distribution sentiment ({n_iterations} itérations)", fontweight="bold")
axes[1].set_xlabel("Score de sentiment")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_coherence.png", dpi=150, bbox_inches="tight")
plt.show()

cv_length = lengths.std() / lengths.mean() * 100
cv_sentiment = sentiments.std() / sentiments.mean() * 100
print(f"\n📊 Coefficient de variation :")
print(f"  • Longueur : {cv_length:.1f}% (seuil : <15%)  {'🟢' if cv_length < 15 else '🟠'}")
print(f"  • Sentiment : {cv_sentiment:.1f}% (seuil : <20%) {'🟢' if cv_sentiment < 20 else '🟠'}")

## 5.3 Score de robustesse

In [ ]:
robustness_score = {
    "perturbations": avg_by_perturbation.mean() * 100,
    "coherence_longueur": max(0, 100 - cv_length * 3),
    "coherence_sentiment": max(0, 100 - cv_sentiment * 2),
    "stabilite_temporelle": 88,  # Simulation
    "tests_limites": 75  # Simulation
}

weights = {
    "perturbations": 0.25,
    "coherence_longueur": 0.20,
    "coherence_sentiment": 0.20,
    "stabilite_temporelle": 0.20,
    "tests_limites": 0.15
}

score_global = sum(robustness_score[k] * weights[k] for k in weights)

if score_global >= 85: grade = "A"
elif score_global >= 70: grade = "B"
elif score_global >= 55: grade = "C"
elif score_global >= 40: grade = "D"
else: grade = "E"

fig, ax = plt.subplots(figsize=(10, 5))
categories = list(robustness_score.keys())
values = list(robustness_score.values())
colors = ["#27ae60" if v >= 70 else "#f39c12" if v >= 50 else "#e74c3c" for v in values]
ax.barh(categories, values, color=colors, edgecolor="black")
ax.set_xlim(0, 100)
ax.set_title(f"Score robustesse : {score_global:.1f}/100 — Grade {grade}", 
             fontsize=14, fontweight="bold")
for i, v in enumerate(values):
    ax.text(v + 1, i, f"{v:.0f}", va="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_robustness_score.png", dpi=150, bbox_inches="tight")
plt.show()

# Sauvegarde
etape5_results = {"robustness_score": robustness_score, "global_score": score_global, "grade": grade}
with open(OUTPUT_DIR / "05_etape5_results.yaml", "w") as f:
    yaml.dump(etape5_results, f, allow_unicode=True)

print(f"\n🎯 SCORE ROBUSTESSE : {score_global:.1f}/100 (Grade {grade})")
print("\n➡️  Notebook suivant : 06_synthese_remediation.ipynb")